# 🛡️ SecAudit Knowledge Base Builder (Colab T4 GPU Edition)
This notebook clones the 12 target repositories, chunks all technical specifications, computes dense embeddings using **Nvidia T4 GPU**, builds the BM25 index, generates `INGESTION_MANIFEST.md`, and exports a compressed bundle directly to your **Google Drive**.

### Step 1: Mount Google Drive
Run this cell and authenticate so the finished bundle saves directly to your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/secaudit_bundle

### Step 2: Install High-Performance Dependencies
Installs ChromaDB, Sentence-Transformers, PyYAML, BM25, and LangChain splitters.

In [ ]:
!pip install -q chromadb sentence-transformers rank-bm25 pyyaml langchain-text-splitters tqdm tabulate pigz

### Step 3: Fast Parallel Shallow Clones of All 12 Repositories

In [ ]:
import os
os.makedirs('/content/raw_repos', exist_ok=True)
%cd /content/raw_repos

repos = [
    ("https://github.com/tldr-pages/tldr.git", "tldr"),
    ("https://github.com/angrave/SystemProgramming.git", "SystemProgramming"),
    ("https://github.com/OWASP/CheatSheetSeries.git", "OWASP-CheatSheets"),
    ("https://github.com/complytime/complytime-policies.git", "complytime"),
    ("https://github.com/ansible-lockdown/UBUNTU22-CIS.git", "ubuntu-cis"),
    ("https://github.com/ansible-lockdown/RHEL9-CIS.git", "rhel-cis"),
    ("https://github.com/trimstray/the-practical-linux-hardening-guide.git", "linux-hardening"),
    ("https://github.com/leandromoreira/linux-network-performance-parameters.git", "network-performance"),
    ("https://github.com/aosabook/aosabook.git", "aosabook"),
    ("https://github.com/donnemartin/system-design-primer.git", "system-design-primer"),
    ("https://github.com/libuv/libuv.git", "libuv"),
    ("https://github.com/dendibakh/perf-ninja.git", "perf-ninja"),
    ("https://github.com/axboe/liburing.git", "liburing"),
    ("https://github.com/electronicarts/EASTL.git", "EASTL")
]

for url, dest in repos:
    if not os.path.exists(dest):
        print(f"Cloning {dest}...")
        !git clone --depth 1 --single-branch {url} {dest}
print("All target repositories fetched!")

### Step 4: Chunking, BM25 Indexing & T4 GPU Batch Embedding

In [ ]:
import os
import gc
import glob
import pickle
import datetime
from collections import defaultdict
from pathlib import Path
from tqdm import tqdm
import yaml
import torch

import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

ROOT_DIR = Path("/content/raw_repos")
OUTPUT_DIR = Path("/content/output")
DB_DIR = OUTPUT_DIR / "chroma_db"
BM25_PATH = OUTPUT_DIR / "bm25_index.pkl"
MANIFEST_PATH = OUTPUT_DIR / "INGESTION_MANIFEST.md"

DB_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Active Acceleration Device: {device.upper()}")
if device == "cuda":
    print(f"[*] GPU Model: {torch.cuda.get_device_name(0)}")

print("Loading BAAI/bge-small-en-v1.5 onto GPU...")
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)

chroma_client = chromadb.PersistentClient(path=str(DB_DIR))
collection = chroma_client.get_or_create_collection(
    name="secaudit_corpus",
    metadata={"hnsw:space": "cosine"}
)

documents = []
metadatas = []
ids = []
file_manifest = []
category_counter = defaultdict(int)
repo_counter = defaultdict(lambda: {"files": 0, "chunks": 0})

headers_to_split_on = [("#", "header_1"), ("##", "header_2"), ("###", "header_3")]
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on, strip_headers=False)
char_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=120)
doc_counter = 0

def add_record(text: str, meta: dict):
    global doc_counter
    text = text.strip()
    if len(text) < 50:
        return
    doc_counter += 1
    doc_id = f"SECAUDIT_{doc_counter:07d}"
    documents.append(text)
    cleaned_meta = {k: str(v) for k, v in meta.items()}
    metadatas.append(cleaned_meta)
    ids.append(doc_id)
    category_counter[cleaned_meta.get("category", "Uncategorized")] += 1
    repo_counter[cleaned_meta.get("source_repo", "Unknown")]["chunks"] += 1

def log_file(repo: str, rel_path: str, doc_type: str, title: str, chunks_added: int):
    repo_counter[repo]["files"] += 1
    file_manifest.append({
        "repo": repo,
        "file_name": rel_path,
        "document_type": doc_type,
        "abstract_title": title,
        "chunks": chunks_added
    })

# 1. TLDR Pages
tldr_files = glob.glob(str(ROOT_DIR / "tldr/pages/common/*.md")) + \
             glob.glob(str(ROOT_DIR / "tldr/pages/linux/*.md")) + \
             glob.glob(str(ROOT_DIR / "tldr/pages/osx/*.md"))
for fpath in tqdm(tldr_files, desc="TLDR"):
    try:
        content = Path(fpath).read_text(encoding="utf-8", errors="ignore")
        cmd = Path(fpath).stem
        rel = str(Path(fpath).relative_to(ROOT_DIR / "tldr"))
        b = doc_counter
        add_record(content, {"source_repo": "tldr-pages/tldr", "category": "POSIX Command Reference", "file_name": rel, "title": f"Command: {cmd}", "target_agent": "systems_architect"})
        if doc_counter - b > 0:
            log_file("tldr-pages/tldr", rel, "Command Specification", f"Command Reference ({cmd})", doc_counter - b)
    except Exception:
        continue

# 2. OWASP Cheat Sheets
owasp_files = glob.glob(str(ROOT_DIR / "OWASP-CheatSheets/cheatsheets/*.md"))
for fpath in tqdm(owasp_files, desc="OWASP"):
    try:
        content = Path(fpath).read_text(encoding="utf-8", errors="ignore")
        splits = char_splitter.split_documents(md_splitter.split_text(content))
        title = Path(fpath).stem.replace("_", " ")
        rel = str(Path(fpath).relative_to(ROOT_DIR / "OWASP-CheatSheets"))
        b = doc_counter
        for d in splits:
            add_record(d.page_content, {"source_repo": "OWASP/CheatSheetSeries", "category": "Application Security Standard", "file_name": rel, "title": f"OWASP Standard: {title}", "target_agent": "security_auditor", **d.metadata})
        if doc_counter - b > 0:
            log_file("OWASP/CheatSheetSeries", rel, "Security Benchmark", f"Security Guide ({title})", doc_counter - b)
    except Exception:
        continue

# 3. System Programming (angrave)
for fpath in glob.glob(str(ROOT_DIR / "SystemProgramming/**/*.md"), recursive=True):
    try:
        content = Path(fpath).read_text(encoding="utf-8", errors="ignore")
        splits = char_splitter.split_documents(md_splitter.split_text(content))
        title = Path(fpath).stem.replace("_", " ")
        rel = str(Path(fpath).relative_to(ROOT_DIR / "SystemProgramming"))
        b = doc_counter
        for d in splits:
            add_record(d.page_content, {"source_repo": "angrave/SystemProgramming", "category": "POSIX Concurrency & OS Specs", "file_name": rel, "title": f"POSIX Arch: {title}", "target_agent": "systems_architect", **d.metadata})
        if doc_counter - b > 0:
            log_file("angrave/SystemProgramming", rel, "Academic Systems Spec", f"POSIX Architecture ({title})", doc_counter - b)
    except Exception:
        continue

# 4. Linux Hardening & Network Parameters
hardening_files = [
    ("trimstray/linux-hardening", glob.glob(str(ROOT_DIR / "linux-hardening/**/*.md"), recursive=True), "Host & Kernel Hardening"),
    ("leandromoreira/network-perf", glob.glob(str(ROOT_DIR / "network-performance/**/*.md"), recursive=True), "TCP/IP & Socket Tuning")
]
for repo_name, files, cat_title in hardening_files:
    for fpath in files:
        try:
            content = Path(fpath).read_text(encoding="utf-8", errors="ignore")
            splits = char_splitter.split_documents(md_splitter.split_text(content))
            fname = Path(fpath).name
            b = doc_counter
            for d in splits:
                add_record(d.page_content, {"source_repo": repo_name, "category": cat_title, "file_name": fname, "title": f"Hardening: {fname}", "target_agent": "optimizer", **d.metadata})
            if doc_counter - b > 0:
                log_file(repo_name, fname, "Hardening Specification", f"OS Tuning ({fname})", doc_counter - b)
        except Exception:
            continue

# 5. CIS Benchmarks
cis_files = [
    ("ansible-lockdown/UBUNTU22-CIS", glob.glob(str(ROOT_DIR / "ubuntu-cis/tasks/**/*.yml"), recursive=True)),
    ("ansible-lockdown/RHEL9-CIS", glob.glob(str(ROOT_DIR / "rhel-cis/tasks/**/*.yml"), recursive=True)),
    ("complytime/complytime-policies", glob.glob(str(ROOT_DIR / "complytime/**/*.yaml"), recursive=True))
]
for repo_name, files in cis_files:
    for ypath in files:
        try:
            with open(ypath, "r", encoding="utf-8", errors="ignore") as yf:
                parsed = yaml.safe_load(yf)
                if isinstance(parsed, list):
                    fname = Path(ypath).name
                    b = doc_counter
                    for task in parsed:
                        if isinstance(task, dict) and "name" in task:
                            tname = task.get("name", "CIS Task")
                            add_record(f"CIS Policy: {tname}\nAction Config:\n" + yaml.dump(task), {"source_repo": repo_name, "category": "CIS Compliance Remediation", "file_name": fname, "title": tname[:80], "target_agent": "security_auditor"})
                    if doc_counter - b > 0:
                        log_file(repo_name, fname, "CIS Remediation Task Group", f"Compliance Policy ({fname})", doc_counter - b)
        except Exception:
            continue

# 6. Low-Level Performance & Architecture Patterns
code_files = [
    ("dendibakh/perf-ninja", glob.glob(str(ROOT_DIR / "perf-ninja/**/*.md"), recursive=True) + glob.glob(str(ROOT_DIR / "perf-ninja/**/*.cpp"), recursive=True), "CPU Microarchitecture & Vectorization"),
    ("axboe/liburing", glob.glob(str(ROOT_DIR / "liburing/man/*"), recursive=True) + glob.glob(str(ROOT_DIR / "liburing/examples/*.c"), recursive=True), "Modern Async I/O (io_uring)"),
    ("aosabook/aosabook", glob.glob(str(ROOT_DIR / "aosabook/**/*.markdown"), recursive=True), "Production Architecture Patterns"),
    ("donnemartin/system-design-primer", glob.glob(str(ROOT_DIR / "system-design-primer/**/*.md"), recursive=True), "Distributed System Scaling"),
    ("electronicarts/EASTL", glob.glob(str(ROOT_DIR / "EASTL/doc/*.md"), recursive=True), "Cache-Friendly Data Structures")
]
for repo_name, files, cat_title in code_files:
    for fpath in files:
        try:
            content = Path(fpath).read_text(encoding="utf-8", errors="ignore")
            splits = char_splitter.split_text(content)
            fname = Path(fpath).name
            b = doc_counter
            for chunk in splits:
                add_record(chunk, {"source_repo": repo_name, "category": cat_title, "file_name": fname, "title": f"Pattern: {fname}", "target_agent": "optimizer"})
            if doc_counter - b > 0:
                log_file(repo_name, fname, "Architecture & Optimization Pattern", f"Modernization Guideline ({fname})", doc_counter - b)
        except Exception:
            continue

total_docs = len(documents)
print(f"\n>>> Total Extracted Chunks: {total_docs:,}")
print(f">>> Total Files Logged:     {len(file_manifest):,}")

# Build Ingestion Manifest
with open(MANIFEST_PATH, "w", encoding="utf-8") as mf:
    mf.write("# 🛡️ SecAudit Knowledge Base Ingestion Manifest\n\n")
    mf.write(f"**Generated on:** {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    mf.write(f"**Total Ingested Chunks:** `{total_docs:,}`\n")
    mf.write(f"**Total Source Files Parsed:** `{len(file_manifest):,}`\n")
    mf.write("**Embedding Model:** `BAAI/bge-small-en-v1.5` (Cosine Metric on GPU)\n")
    mf.write("**Sparse Indexing:** `BM25Okapi`\n\n")
    mf.write("## 1. Summary by Repository\n\n| Repository | Files | Chunks |\n| :--- | :---: | :---: |\n")
    for repo, stats in sorted(repo_counter.items()):
        mf.write(f"| `{repo}` | {stats['files']:,} | {stats['chunks']:,} |\n")
    mf.write("\n## 2. Summary by Knowledge Domain\n\n| Domain / Category | Chunks |\n| :--- | :---: |\n")
    for cat, count in sorted(category_counter.items(), key=lambda x: x[1], reverse=True):
        mf.write(f"| **{cat}** | {count:,} |\n")
    mf.write("\n## 3. Comprehensive File Manifest\n\n| Repository | File | Type | Title | Chunks |\n| :--- | :--- | :--- | :--- | :---: |\n")
    for f in file_manifest:
        mf.write(f"| `{f['repo']}` | `{f['file_name']}` | {f['document_type']} | {f['abstract_title']} | {f['chunks']} |\n")

# BM25 Indexing
print("Building BM25 Sparse Index...")
tokenized = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized)
with open(BM25_PATH, "wb") as f:
    pickle.dump({"bm25": bm25, "ids": ids, "documents": documents, "metadatas": metadatas}, f, protocol=pickle.HIGHEST_PROTOCOL)
del tokenized
gc.collect()

# GPU Batch Vectorization & ChromaDB Bulk Ingestion
print("Embedding on GPU & streaming to ChromaDB...")
BATCH = 512
with torch.inference_mode():
    for i in tqdm(range(0, total_docs, BATCH), desc="ChromaDB + GPU"):
        end_idx = min(i + BATCH, total_docs)
        b_docs = documents[i:end_idx]
        b_meta = metadatas[i:end_idx]
        b_ids = ids[i:end_idx]
        embeddings = embed_model.encode(b_docs, batch_size=256, show_progress_bar=False, normalize_embeddings=True).tolist()
        collection.add(ids=b_ids, documents=b_docs, metadatas=b_meta, embeddings=embeddings)

print("Ingestion complete! All records persisted.")

### Step 5: Bundle and Export Directly to Google Drive
Creates a compressed `secaudit_knowledge_pack.tar.gz` and copies both the archive and `INGESTION_MANIFEST.md` directly into your Google Drive folder: `secaudit_bundle/`.

In [ ]:
%cd /content/output
!tar -I pigz -cvf secaudit_knowledge_pack.tar.gz chroma_db bm25_index.pkl INGESTION_MANIFEST.md

!cp secaudit_knowledge_pack.tar.gz /content/drive/MyDrive/secaudit_bundle/
!cp INGESTION_MANIFEST.md /content/drive/MyDrive/secaudit_bundle/

print("🎉 SUCCESS! Bundle exported to Google Drive: MyDrive/secaudit_bundle/")
!ls -lh /content/drive/MyDrive/secaudit_bundle/